# 20D activation ablation — fixed-checkpoint evaluation

**Evaluation only; no training.** Run this notebook from its containing directory. It verifies and evaluates the six bundled step-25,000 checkpoints without selecting a seed, checkpoint, or Talbot degree from the observed results. Degree 8 is primary; Degree 5 is diagnostic.


## 1. Imports and environment


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import csv
import gc
import hashlib
import json
import math
import platform
import time
import uuid
import warnings

import numpy as np
import mpmath as mp
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

if mp.__version__ != "1.3.0":
    raise RuntimeError("This evaluation protocol uses mpmath 1.3.0. Install the supplied requirements.")
print({"torch": str(torch.__version__), "mpmath": mp.__version__})


## 2. Paths and fixed evaluation protocol


In [ ]:
# Run from this notebook's directory. No training mode exists.
ARTIFACT_VERSION = "2026-09-07-v3-windows-safe-numpy-tanh"
RUN_MODE = "evaluate"          # "check" validates six models only; "evaluate" also evaluates them
PACKAGE_DIR = Path.cwd()       # directory containing checkpoint_manifest.json
DEVICE = "cpu"                 # audited default; GPU complex tanh may crash at extreme Talbot nodes

if RUN_MODE not in {"check", "evaluate"}:
    raise ValueError("Only check/evaluate modes are supported; this notebook never trains.")
PACKAGE_DIR = PACKAGE_DIR.expanduser().resolve()
if not (PACKAGE_DIR / "checkpoint_manifest.json").is_file():
    raise FileNotFoundError(f"Set PACKAGE_DIR to the extracted folder containing checkpoint_manifest.json. Current: {PACKAGE_DIR}")
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError("DEVICE must be 'cpu' or 'cuda'. The audited default is CPU.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' requested but CUDA is unavailable.")
device = torch.device(DEVICE)
r_dtype, c_dtype = torch.float64, torch.complex128
torch.set_default_dtype(r_dtype)
if device.type == "cpu":
    torch.set_num_threads(1)  # small inference batches; no training
DIMENSION, CHECKPOINT_STEP = 20, 25_000
SEEDS, DEGREES, T_VALUES = (12345, 22345, 32345), (5, 8), tuple(range(1, 11))
MODEL_CONFIG = dict(d=20, coord_embed_dim=16, boundary_embed_dim=16,
                    interior_hidden_width=128, boundary_hidden_width=128,
                    pair_chunk_size=32768, boundary_anchor_mode="hard_rbm_flow_balance")
SAMPLER_CONFIG = {
    "real_domain": [-0.5, 5.0], "imaginary_abs_max": 5.0,
    "bar_mixture": {"general_complex": 0.5, "exact_complex_diagonal": 0.2,
                    "near_complex_diagonal": 0.2, "positive_real_diagonal": 0.1},
    "near_diagonal_noise_scale": 0.05,
}
MANIFEST = json.loads((PACKAGE_DIR / "checkpoint_manifest.json").read_text())
CHECKPOINTS = MANIFEST["checkpoints"]
expected = {(method, seed) for method in ("Cauchy", "tanh") for seed in SEEDS}
actual = [(entry["method"], entry["seed"]) for entry in CHECKPOINTS]
if len(actual) != 6 or set(actual) != expected or MANIFEST["endpoint_step"] != CHECKPOINT_STEP:
    raise ValueError("Manifest must contain exactly the six specified method/seed pairs at step 25000.")
print({"artifact_version": ARTIFACT_VERSION, "mode": RUN_MODE,
       "device": str(device), "checkpoints": actual})


## 3. Compatible model definitions

Definitions are copied from the supplied main and ablation notebooks, retaining
the original state-dict keys and full forward path. Training, optimizers, sampling,
and training diagnostics are not executed. Derivative methods are retained for
the removable zero-limit of the tail transform.


In [ ]:
class CauchyComplex(nn.Module):
    def __init__(self, num_features, init_lambda=0.1, init_d=0.5, d_min=1e-3):
        super().__init__()
        self.num_features = num_features
        self.d_min = d_min
        self.lambda1 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        self.lambda2 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        init_value = max(init_d - d_min, 1e-6)
        init_rho = torch.log(torch.expm1(torch.tensor(init_value, dtype=r_dtype)))
        self.rho = nn.Parameter(torch.ones(num_features, dtype=r_dtype) * init_rho)
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def positive_d(self):
        return F.softplus(self.rho) + self.d_min

    def reset_denominator_tracking(self):
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def _update_denominator_tracking(self, denom):
        current = torch.min(torch.abs(denom)).detach()
        self.last_min_abs_denominator = current
        if self.running_min_abs_denominator is None:
            self.running_min_abs_denominator = current
        else:
            self.running_min_abs_denominator = torch.minimum(self.running_min_abs_denominator, current).detach()

    def _denominator(self, z):
        d_complex = self.positive_d().to(device=z.device).to(dtype=z.dtype)
        return z ** 2 + d_complex ** 2, d_complex

    def forward(self, z):
        denom, _ = self._denominator(z)
        self._update_denominator_tracking(denom)
        out = (self.lambda1 * z + self.lambda2) / denom
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError("Non-finite Cauchy activation output.")
        return out

    def derivative(self, z):
        denom, d_complex = self._denominator(z)
        self._update_denominator_tracking(denom)
        out = (self.lambda1 * (d_complex ** 2 - z ** 2) - 2.0 * self.lambda2 * z) / (denom ** 2)
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError("Non-finite Cauchy derivative output.")
        return out

class CoordinateCauchyScalarNet(nn.Module):
    """
    Shared scalar complex network:
        Linear(input_dim,H) -> CauchyComplex(H) -> Linear(H,1).
    First input column is the scalar coordinate theta_i.
    """

    def __init__(self, input_dim, hidden_width):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_width = hidden_width
        self.linear_0 = nn.Linear(input_dim, hidden_width, dtype=c_dtype)
        self.cauchy_0 = CauchyComplex(hidden_width)
        self.linear_1 = nn.Linear(hidden_width, 1, dtype=c_dtype)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError(f"x must have shape [N,{self.input_dim}], got {tuple(x.shape)}")
        x = x.to(dtype=c_dtype, device=device)
        return self.linear_1(self.cauchy_0(self.linear_0(x)))

    def forward_with_first_input_derivative(self, x):
        x = x.to(dtype=c_dtype, device=device)
        z = self.linear_0(x)                    # [N,H]
        phi = self.cauchy_0(z)                  # [N,H]
        out = self.linear_1(phi)                # [N,1]
        phi_prime = self.cauchy_0.derivative(z) # [N,H]
        # derivative wrt x[:,0]: sum_h W_out[0,h] Phi'_h W_in[h,0]
        deriv = torch.einsum("h,nh,h->n", self.linear_1.weight[0], phi_prime, self.linear_0.weight[:, 0]).unsqueeze(1)
        return out, deriv

def run_scalar_net_chunked(net, x, pair_chunk_size=32768, with_derivative=False):
    outs = []
    derivs = []
    for start in range(0, x.shape[0], pair_chunk_size):
        chunk = x[start:start + pair_chunk_size]
        if with_derivative:
            out, deriv = net.forward_with_first_input_derivative(chunk)
            outs.append(out)
            derivs.append(deriv)
        else:
            outs.append(net(chunk))
    if with_derivative:
        return torch.cat(outs, dim=0), torch.cat(derivs, dim=0)
    return torch.cat(outs, dim=0)

class StructuredCoordinateCauchyMGF(nn.Module):
    """
    Coordinate-wise additive Cauchy model.
    Python k is zero-based and represents mathematical boundary k+1.
    Boundary terms are zero-centered so f_k(0)=log(boundary_mass[k]).
    """

    def __init__(
        self,
        d=20,
        coord_embed_dim=16,
        boundary_embed_dim=16,
        interior_hidden_width=128,
        boundary_hidden_width=128,
        pair_chunk_size=32768,
        boundary_mass=None,
    ):
        super().__init__()
        self.d = int(d)
        self.coord_embed_dim = int(coord_embed_dim)
        self.boundary_embed_dim = int(boundary_embed_dim)
        self.pair_chunk_size = int(pair_chunk_size)
        if boundary_mass is None:
            raise ValueError("StructuredCoordinateCauchyMGF requires boundary_mass.")
        boundary_mass_real = boundary_mass.detach().clone().to(dtype=r_dtype, device=device)
        if boundary_mass_real.shape != (self.d,):
            raise ValueError(f"boundary_mass must have shape ({self.d},), got {tuple(boundary_mass_real.shape)}")
        if not torch.isfinite(boundary_mass_real).all():
            raise ValueError("boundary_mass contains non-finite entries.")
        if torch.any(boundary_mass_real <= 0):
            raise ValueError("boundary_mass entries must be strictly positive.")
        self.register_buffer("boundary_mass", boundary_mass_real, persistent=True)
        self.register_buffer("boundary_log_mass", torch.log(boundary_mass_real).to(c_dtype), persistent=True)
        scale = 0.01
        self.coord_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype),
                torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype),
            )
        )
        self.boundary_embedding = nn.Parameter(
            scale * torch.complex(
                torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype),
                torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype),
            )
        )
        self.interior_net = CoordinateCauchyScalarNet(1 + self.coord_embed_dim, interior_hidden_width)
        self.boundary_net = CoordinateCauchyScalarNet(1 + self.coord_embed_dim + self.boundary_embed_dim, boundary_hidden_width)

    def _interior_inputs(self, theta):
        B, d = theta.shape
        coord = self.coord_embedding.view(1, d, self.coord_embed_dim).expand(B, d, self.coord_embed_dim)
        return torch.cat([theta.unsqueeze(-1), coord], dim=2).reshape(B * d, 1 + self.coord_embed_dim)

    def _interior_f0_and_jac(self, theta, need_jacobian):
        B, d = theta.shape
        x = self._interior_inputs(theta)
        if need_jacobian:
            q, dq = run_scalar_net_chunked(self.interior_net, x, self.pair_chunk_size, with_derivative=True)
            dq = dq.view(B, d)
        else:
            q = run_scalar_net_chunked(self.interior_net, x, self.pair_chunk_size, with_derivative=False)
            dq = None
        q = q.view(B, d)
        zero_theta = torch.zeros(d, 1, dtype=c_dtype, device=theta.device)
        x0 = torch.cat([zero_theta, self.coord_embedding], dim=1)
        q0 = run_scalar_net_chunked(self.interior_net, x0, self.pair_chunk_size, with_derivative=False).view(1, d)
        f0 = torch.sum(q - q0, dim=1, keepdim=True)
        return f0, dq

    def _boundary_inputs_for_k(self, theta, k):
        B, d = theta.shape
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        theta_sel = theta[:, mask]
        coord = self.coord_embedding[mask].view(1, d - 1, self.coord_embed_dim).expand(B, d - 1, self.coord_embed_dim)
        bd = self.boundary_embedding[k].view(1, 1, self.boundary_embed_dim).expand(B, d - 1, self.boundary_embed_dim)
        x = torch.cat([theta_sel.unsqueeze(-1), coord, bd], dim=2).reshape(B * (d - 1), 1 + self.coord_embed_dim + self.boundary_embed_dim)
        return x, mask

    def _boundary_zero_inputs_for_k(self, theta, k):
        d = self.d
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        zeros = torch.zeros(d - 1, 1, dtype=c_dtype, device=theta.device)
        coord = self.coord_embedding[mask]
        bd = self.boundary_embedding[k].view(1, self.boundary_embed_dim).expand(d - 1, self.boundary_embed_dim)
        return torch.cat([zeros, coord, bd], dim=1)

    def _boundary_fk_and_deriv_for_k(self, theta, k, need_derivative):
        B, d = theta.shape
        x, mask = self._boundary_inputs_for_k(theta, k)
        x0 = self._boundary_zero_inputs_for_k(theta, k)
        if need_derivative:
            q, dq = run_scalar_net_chunked(self.boundary_net, x, self.pair_chunk_size, with_derivative=True)
            dq = dq.view(B, d - 1)
        else:
            q = run_scalar_net_chunked(self.boundary_net, x, self.pair_chunk_size, with_derivative=False)
            dq = None
        q0 = run_scalar_net_chunked(self.boundary_net, x0, self.pair_chunk_size, with_derivative=False).view(1, d - 1)
        q = q.view(B, d - 1)
        fk = self.boundary_log_mass[k].view(1, 1) + torch.sum(q - q0, dim=1, keepdim=True)
        return fk, dq, mask

    def forward_selected_boundary(self, theta, k):
        theta = theta.to(dtype=c_dtype, device=device)
        if k < 0 or k >= self.d:
            raise IndexError(f"Boundary index k={k} outside [0,{self.d-1}]")
        f0, _ = self._interior_f0_and_jac(theta, need_jacobian=False)
        fk, _, _ = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=False)
        return f0, fk

    def forward(self, theta):
        f_mat, _ = self.forward_with_input_jacobian(theta, need_jacobian=False)
        return f_mat

    def forward_with_input_jacobian(self, theta, need_jacobian=True):
        theta = theta.to(dtype=c_dtype, device=device)
        if theta.ndim != 2 or theta.shape[1] != self.d:
            raise ValueError(f"theta must have shape [B,{self.d}], got {tuple(theta.shape)}")
        B, d = theta.shape
        f0, dq_int = self._interior_f0_and_jac(theta, need_jacobian=need_jacobian)
        boundary_cols = []
        if need_jacobian:
            J = torch.zeros(B, d + 1, d, dtype=c_dtype, device=theta.device)
            J[:, 0, :] = dq_int
        else:
            J = None
        for k in range(d):
            fk, dq, mask = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=need_jacobian)
            boundary_cols.append(fk)
            if need_jacobian:
                J[:, k + 1, mask] = dq
        return torch.cat([f0] + boundary_cols, dim=1), J

class ComplexTanh(nn.Module):
    def __init__(self, num_features):
        super().__init__(); self.num_features=int(num_features); self.last_stats=None
    def forward(self,z):
        out=torch.tanh(z)
        self.last_stats={"z_real_min":float(z.real.min().detach().cpu()),"z_real_max":float(z.real.max().detach().cpu()),"z_imag_min":float(z.imag.min().detach().cpu()),"z_imag_max":float(z.imag.max().detach().cpu()),"max_abs_tanh":float(out.abs().max().detach().cpu())}
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError(f"Non-finite complex tanh; stats={self.last_stats}")
        return out
    def derivative(self,z):
        out=1-torch.tanh(z)**2
        if not torch.isfinite(out.real).all() or not torch.isfinite(out.imag).all():
            raise FloatingPointError(f"Non-finite complex tanh derivative; stats={self.last_stats}")
        return out

class CoordinateTanhScalarNet(nn.Module):
    def __init__(self,input_dim,hidden_width):
        super().__init__(); self.input_dim=input_dim; self.hidden_width=hidden_width
        self.linear_0=nn.Linear(input_dim,hidden_width,dtype=c_dtype)
        self.tanh_0=ComplexTanh(hidden_width)
        self.linear_1=nn.Linear(hidden_width,1,dtype=c_dtype)
    def forward(self,x): return self.linear_1(self.tanh_0(self.linear_0(x.to(dtype=c_dtype,device=device))))
    def forward_with_first_input_derivative(self,x):
        x=x.to(dtype=c_dtype,device=device); z=self.linear_0(x); phi=self.tanh_0(z); out=self.linear_1(phi)
        deriv=torch.einsum("h,nh,h->n",self.linear_1.weight[0],self.tanh_0.derivative(z),self.linear_0.weight[:,0]).unsqueeze(1)
        return out,deriv

class StructuredCoordinateTanhMGF(StructuredCoordinateCauchyMGF):
    def __init__(self,d=20,coord_embed_dim=16,boundary_embed_dim=16,interior_hidden_width=128,boundary_hidden_width=128,pair_chunk_size=32768,boundary_mass=None):
        nn.Module.__init__(self)
        self.d=int(d); self.coord_embed_dim=int(coord_embed_dim); self.boundary_embed_dim=int(boundary_embed_dim); self.pair_chunk_size=int(pair_chunk_size)
        bm=boundary_mass.detach().clone().to(dtype=r_dtype,device=device)
        if bm.shape!=(self.d,) or (not torch.isfinite(bm).all()) or torch.any(bm<=0): raise ValueError("invalid boundary_mass")
        self.register_buffer("boundary_mass",bm,persistent=True); self.register_buffer("boundary_log_mass",torch.log(bm).to(c_dtype),persistent=True)
        scale=0.01
        self.coord_embedding=nn.Parameter(scale*torch.complex(torch.randn(self.d,self.coord_embed_dim,dtype=r_dtype),torch.randn(self.d,self.coord_embed_dim,dtype=r_dtype)))
        self.boundary_embedding=nn.Parameter(scale*torch.complex(torch.randn(self.d,self.boundary_embed_dim,dtype=r_dtype),torch.randn(self.d,self.boundary_embed_dim,dtype=r_dtype)))
        self.interior_net=CoordinateTanhScalarNet(1+self.coord_embed_dim,interior_hidden_width)
        self.boundary_net=CoordinateTanhScalarNet(1+self.coord_embed_dim+self.boundary_embed_dim,boundary_hidden_width)


## 4. Validate and load the six endpoints

Checks: SHA256, top-level seed, step, serialized configuration, state-dict keys,
shapes/dtypes, finite weights, boundary masses, optimizer counters, normalization,
and parameter counts. Only restricted `weights_only=True` loading is allowed.

The two historical tanh 22345/32345 payloads contain a stale nested `config.seed=12345`.
Their top-level seeds were written from the actual training argument; the known
inconsistency is recorded, not silently rewritten. Other mismatches stop evaluation.

Cauchy has 16,388 real parameter degrees of freedom; tanh has 15,108. Tanh
checkpoint tensors are validated by PyTorch and then its frozen diagonal transform
is evaluated with the same complex128 affine–tanh–affine equations in NumPy. This
avoids the observed Windows PyTorch complex-tanh native crash; it does not reuse
archived tanh predictions. Cauchy remains evaluated in PyTorch.

This is a
same-width/same-training-budget comparison, not strictly parameter-matched.


In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def checked_path(relative):
    path = (PACKAGE_DIR / relative).resolve()
    if not path.is_relative_to(PACKAGE_DIR) or not path.is_file():
        raise FileNotFoundError(f"Missing or invalid packaged file: {relative}")
    return path


def require_fields(config, expected, label):
    for key, value in expected.items():
        if key not in config or config[key] != value:
            raise ValueError(f"{label}.{key}: expected {value!r}, got {config.get(key)!r}")


class NumpyTanhDiagonalEvaluator:
    """Evaluate the frozen tanh interior network without calling torch.tanh.

    This is the same complex128 affine-tanh-affine map. The coordinate-shared
    full model defines f0 as the sum of centered interior terms, so boundary
    networks are irrelevant to the diagonal transform used by Talbot.
    """

    backend = "numpy_complex128"

    def __init__(self, state):
        def array(key):
            return np.asarray(state[key].detach().cpu().numpy(), dtype=np.complex128)
        self.coord_embedding = array("coord_embedding")
        self.weight0 = array("interior_net.linear_0.weight")
        self.bias0 = array("interior_net.linear_0.bias")
        self.weight1 = array("interior_net.linear_1.weight")
        self.bias1 = array("interior_net.linear_1.bias")
        self.q_zero = self._q(0j)

    def _q(self, z):
        x = np.concatenate([
            np.full((DIMENSION, 1), np.complex128(z), dtype=np.complex128),
            self.coord_embedding,
        ], axis=1)
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            return np.tanh(x @ self.weight0.T + self.bias0) @ self.weight1.T + self.bias1

    def phi(self, s):
        z = complex(float(mp.re(s)), float(mp.im(s)))
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            value = np.exp(np.sum(self._q(z) - self.q_zero))
        return mp.mpc(float(value.real), float(value.imag))

    def zero_limit(self):
        x0 = np.concatenate([
            np.zeros((DIMENSION, 1), dtype=np.complex128), self.coord_embedding
        ], axis=1)
        z0 = x0 @ self.weight0.T + self.bias0
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            derivative = np.sum(
                (1.0 - np.tanh(z0) ** 2)
                * self.weight0[:, 0][None, :]
                * self.weight1[0][None, :]
            )
        return mp.mpc(float(-derivative.real), float(-derivative.imag))

    def tail_transform(self, s):
        s = mp.mpc(s)
        if s == 0:
            return self.zero_limit()
        return (1 - self.phi(s)) / s


def validate_config(payload, entry):
    method, seed = entry["method"], entry["seed"]
    if payload.get("seed") != seed or payload.get("last_completed_step") != CHECKPOINT_STEP:
        raise ValueError(f"Wrong seed/step for {entry['path']}")
    notes = []
    if entry["format"] == "public":
        if method != "Cauchy" or payload.get("experiment_slug") != "d20_skew_symmetric_product_form":
            raise ValueError("Unexpected public checkpoint experiment.")
        require_fields(payload["model_config"], MODEL_CONFIG, "model_config")
        require_fields(payload["sampler_config"], SAMPLER_CONFIG, "sampler_config")
        require_fields(payload["training_config"], {
            "optimizer": "Adam", "steps": 25000, "batch_bar": 2048,
            "batch_boundary": 256, "batch_monotonicity": 256,
            "lr_phase1": 5e-5, "lr_phase2": 1e-5, "lr_switch_after_step": 20000,
            "preserve_adam_state_at_lr_switch": True, "gradient_clip": 1.0,
            "lambda_bar": 1.0, "lambda_boundary": 1.0, "lambda_origin": 0.0,
            "lambda_monotonicity": 1.0, "lambda_imaginary": 1.0,
            "monotonicity_real_domain": [-0.5, 5.0],
            "adaptive_bar_weight": {"enabled": True, "alpha": 0.3, "epsilon": 1e-12,
                                    "minimum": 0.5, "maximum": 2.0},
        }, "training_config")
    elif entry["format"] == "legacy":
        expected_method = "A_Cauchy" if method == "Cauchy" else "B_complex_tanh"
        if payload.get("method") != expected_method:
            raise ValueError("Legacy checkpoint method mismatch.")
        config = payload["config"]
        require_fields(config, MODEL_CONFIG, "legacy model config")
        require_fields(config, {
            "steps": 25000, "batch_bar": 2048, "batch_bdary": 256, "batch_mono": 256,
            "lr_phase1": 5e-5, "lr_phase2": 1e-5, "lr_switch_after_step": 20000,
            "grad_clip": 1.0, "lambda_bar": 1.0, "lambda_bdary": 1.0,
            "lambda_zero": 0.0, "lambda_mono": 1.0, "lambda_imag": 1.0,
            "mono_re_min": -0.5, "mono_re_max": 5.0,
            "adaptive_bar_weight": True, "bar_weight_alpha": 0.3,
            "bar_weight_eps": 1e-12, "bar_weight_min": 0.5, "bar_weight_max": 2.0,
        }, "legacy training config")
        if config.get("seed") != seed:
            # Historical train_b saved B_CONFIG with a stale nested seed.
            # The top-level seed is written from the actual train_b(seed) argument.
            if method != "tanh" or seed not in (22345, 32345) or config.get("seed") != 12345:
                raise ValueError("Unrecognized nested seed inconsistency.")
            notes.append("Legacy shared config.seed remains 12345; verified top-level seed is authoritative. Original file unchanged.")
        notes.append("Sampler is not serialized in this legacy payload; source-code correspondence was audited separately, not inferred from the weights.")
    else:
        raise ValueError("Unknown checkpoint schema.")
    return notes


def load_fixed_model(entry):
    path = checked_path(entry["path"])
    blob = path.read_bytes()
    before = hashlib.sha256(blob).hexdigest()
    if before != entry["sha256"]:
        raise ValueError(f"SHA256 mismatch; refusing to substitute checkpoint: {entry['path']}")
    # Restricted PyTorch loading only; do not fall back to weights_only=False.
    payload = torch.load(BytesIO(blob), map_location="cpu", weights_only=True)
    notes = validate_config(payload, entry)
    state = payload["model_state_dict"]
    if not all(isinstance(v, torch.Tensor) and torch.isfinite(v).all().item() for v in state.values()):
        raise ValueError("Non-finite or non-tensor model state.")
    expected_mass = torch.arange(1, 21, dtype=r_dtype)
    if not torch.allclose(state["boundary_mass"], expected_mass, atol=1e-12, rtol=0):
        raise ValueError("Boundary masses do not match the 20D benchmark.")
    if not torch.allclose(state["boundary_log_mass"], torch.log(expected_mass).to(c_dtype), atol=1e-12, rtol=0):
        raise ValueError("Boundary log-mass mismatch.")
    optimizer_state = payload["optimizer_state_dict"]["state"]
    optimizer_steps = {float(v["step"].item()) for v in optimizer_state.values()}
    if optimizer_steps != {CHECKPOINT_STEP}:
        raise ValueError(f"Unexpected optimizer step counters: {optimizer_steps}")
    config = {k: v for k, v in MODEL_CONFIG.items() if k != "boundary_anchor_mode"}
    cls = StructuredCoordinateCauchyMGF if entry["method"] == "Cauchy" else StructuredCoordinateTanhMGF
    model_template = cls(**config, boundary_mass=expected_mass.to(device)).to(device)
    template = model_template.state_dict()
    if set(state) != set(template):
        raise ValueError("Unexpected/missing state-dict keys.")
    if any(state[k].shape != template[k].shape or state[k].dtype != template[k].dtype for k in state):
        raise ValueError("State-dict shape/dtype mismatch; no silent casts allowed.")
    model_template.load_state_dict(state, strict=True)
    model_template.eval()
    model_template.requires_grad_(False)
    real_dof = sum(p.numel() * (2 if p.is_complex() else 1) for p in model_template.parameters())
    expected_dof = 16388 if entry["method"] == "Cauchy" else 15108
    if real_dof != expected_dof:
        raise ValueError("Unexpected parameter budget.")
    if entry["method"] == "Cauchy":
        with torch.no_grad():
            at_zero = torch.exp(model_template(torch.zeros(1, 20, dtype=c_dtype, device=device))[0])
            targets = torch.cat([torch.ones(1, dtype=r_dtype, device=device), expected_mass.to(device)]).to(c_dtype)
            if not torch.allclose(at_zero, targets, atol=1e-10, rtol=1e-12):
                raise ValueError("Hard-normalization check failed.")
        model = model_template
        evaluation_backend = "torch_complex128"
    else:
        # Do not call torch.tanh: the supplied Windows environment repeatedly
        # terminated its Jupyter kernel at extreme Talbot nodes without a Python exception.
        model = NumpyTanhDiagonalEvaluator(state)
        at_zero = model.phi(mp.mpf("0"))
        if at_zero != mp.mpc(1):
            raise ValueError("Tanh origin normalization check failed.")
        evaluation_backend = model.backend
        del model_template
    audit = dict(method=entry["method"], seed=entry["seed"], step=CHECKPOINT_STEP,
                 checkpoint=entry["path"], sha256_before=before, real_parameter_dof=real_dof,
                 optimizer_steps=sorted(optimizer_steps), evaluation_backend=evaluation_backend,
                 notes=notes)
    return model, audit


def validate_all_checkpoints():
    audits = []
    for entry in CHECKPOINTS:
        model, audit = load_fixed_model(entry)
        audits.append(audit)
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        print(f"Checked {entry['method']} seed {entry['seed']}: step 25000")
    return audits


PREFLIGHT = validate_all_checkpoints()


## 5. Evaluate, aggregate, and save / 自动计算并保存

For the diagonal Laplace transform $\phi(s)$, invert $(1-\phi(s))/s$ using
stock `mpmath.invertlaplace(method="talbot", degree=5/8)` at $t=1,\ldots,10$.
The benchmark is the sum of independent exponential variables with rates 1–20.
The analytic-transform diagnostic measures inversion error on that transform;
it is **not** a lower bound on the learned model's error (errors can cancel).

Degree 8 is the primary protocol; degree 5 is retained as a diagnostic/comparison
bridge. Do not change degrees or select seeds after seeing these outputs.
Outer precision is 100 decimal digits, but stock Talbot chooses internal precision
from degree; the neural evaluation remains complex128.

All finite predictions, including negative or above-one values, enter MRE without
clipping. A nonfinite error makes that interval's MRE undefined; it is not dropped.
Summary columns ending in `_percent` are percentages; `mre`/`relative_error` are
fractions. Sample SD uses `ddof=1`, in percentage points.

The NN4MGF reference is **archived**, not a newly evaluated third model. Its checkpoint
was not supplied. Its summary is saved separately and it is not pooled over seeds.


In [ ]:
# Same stock mpmath Talbot call as the supplied notebooks.
# mp.workdps(100) sets OUTER precision. degree overrides Talbot's internal precision.
# Model evaluations are complex128, not 100-digit neural inference.
with mp.workdps(100):
    RATES = [mp.mpf(i) for i in range(1, 21)]
    TRUE_TAIL = {}
    for t in T_VALUES:
        terms = []
        for i, rate in enumerate(RATES):
            coefficient = mp.mpf(1)
            for j, other in enumerate(RATES):
                if i != j:
                    coefficient *= other / (other - rate)
            terms.append(coefficient * mp.exp(-rate * t))
        TRUE_TAIL[t] = float(sum(terms, mp.mpf(0)))


def exact_tail_transform(s):
    s = mp.mpc(s)
    if s == 0:
        return mp.fsum(1 / r for r in RATES)
    value = mp.mpf(1)
    for rate in RATES:
        value *= rate / (rate + s)
    return (1 - value) / s


def learned_tail_transform(model):
    if isinstance(model, NumpyTanhDiagonalEvaluator):
        return model.tail_transform

    @torch.no_grad()
    def transform(s):
        z = mp.mpc(s)
        theta = torch.full((1, 20), complex(float(mp.re(z)), float(mp.im(z))),
                           dtype=c_dtype, device=device)
        if z == 0:
            _, jac = model.forward_with_input_jacobian(theta, need_jacobian=True)
            value = -torch.sum(jac[0, 0, :])
            return mp.mpc(float(value.real.cpu()), float(value.imag.cpu()))
        # Keep the original full forward path for comparison to historical runs.
        value = torch.exp(model(theta)[0, 0])
        if not torch.isfinite(value).item():
            raise FloatingPointError("Non-finite learned transform.")
        phi = mp.mpc(float(value.real.cpu()), float(value.imag.cpu()))
        return (1 - phi) / z
    return transform


def evaluate_transform(method, seed, transform):
    rows = []
    for degree in DEGREES:
        for t in T_VALUES:
            error_message = ""
            try:
                with mp.workdps(100):
                    value = mp.invertlaplace(transform, mp.mpf(t), method="talbot", degree=degree)
                    prediction, imaginary = float(mp.re(value)), float(mp.im(value))
            except (FloatingPointError, OverflowError, ZeroDivisionError) as exc:
                prediction, imaginary = float("nan"), float("nan")
                error_message = f"{type(exc).__name__}: {exc}"
            finite = math.isfinite(prediction) and math.isfinite(imaginary)
            invalid = not finite or abs(imaginary) > 1e-8 or prediction < 0 or prediction > 1
            relative = abs(prediction - TRUE_TAIL[t]) / abs(TRUE_TAIL[t])
            rows.append(dict(method=method, seed=seed, step=CHECKPOINT_STEP if method != "exact_transform" else "",
                             degree=degree, t=t, true_value=TRUE_TAIL[t], raw_prediction=prediction,
                             imaginary_residual=imaginary, relative_error=relative,
                             relative_error_percent=100 * relative, invalid_prediction=int(invalid),
                             nonfinite_prediction=int(not finite), evaluation_error=error_message,
                             evaluation_origin="recomputed"))
    return rows


def summarize(rows):
    summary = []
    groups = sorted({(r["method"], str(r["seed"]), r["degree"]) for r in rows})
    for method, seed, degree in groups:
        group = [r for r in rows if (r["method"], str(r["seed"]), r["degree"]) == (method, seed, degree)]
        for end in (5, 8, 10):
            selected = [r for r in group if r["t"] <= end]
            if len(selected) != end or {r["t"] for r in selected} != set(range(1, end + 1)):
                raise ValueError("Incomplete/duplicate evaluation grid; refusing to average.")
            errors = [r["relative_error"] for r in selected]
            complete = all(math.isfinite(e) for e in errors)
            # Finite out-of-range predictions remain in MRE. Never nanmean/drop failures.
            mre = float(np.mean(errors)) if complete else float("nan")
            summary.append(dict(method=method, seed=seed, degree=degree, interval=f"1:{end}",
                                n_points=end, mre=mre, mre_percent=100 * mre,
                                invalid_count=sum(r["invalid_prediction"] for r in selected),
                                nonfinite_count=sum(r["nonfinite_prediction"] for r in selected),
                                metric_status="complete" if complete else "undefined_nonfinite_error",
                                evaluation_origin=group[0]["evaluation_origin"]))
    return summary


def aggregate(summary):
    rows = []
    for method in ("Cauchy", "tanh"):
        for degree in DEGREES:
            for interval in ("1:5", "1:8", "1:10"):
                group = [r for r in summary if (r["method"], r["degree"], r["interval"]) == (method, degree, interval)]
                if len(group) != 3 or {int(r["seed"]) for r in group} != set(SEEDS):
                    raise ValueError("All three prescribed seeds must be present; no seed selection.")
                values = [r["mre"] for r in group]
                complete = all(math.isfinite(v) for v in values)
                mean = float(np.mean(values)) if complete else float("nan")
                sd = float(np.std(values, ddof=1)) if complete else float("nan")
                rows.append(dict(method=method, degree=degree, interval=interval, n_seeds=3,
                                 mean_mre_percent=100 * mean, sample_sd_percentage_points=100 * sd,
                                 sd_ddof=1, invalid_count=sum(r["invalid_count"] for r in group),
                                 nonfinite_count=sum(r["nonfinite_count"] for r in group),
                                 metric_status="complete" if complete else "undefined_nonfinite_error"))
    return rows


def archived_baseline():
    path = checked_path("reference/nn4mgf_archived_pointwise.csv")
    provenance = json.loads(checked_path("reference/provenance.json").read_text())
    if sha256_file(path) != provenance["csv_sha256"]:
        raise ValueError("Archived baseline hash mismatch.")
    with path.open(newline="", encoding="utf-8-sig") as stream:
        original = list(csv.DictReader(stream))
    if len(original) != 20 or {(int(r["degree"]), int(r["t"])) for r in original} != {(d,t) for d in DEGREES for t in T_VALUES}:
        raise ValueError("Archived baseline grid mismatch.")
    rows = []
    for r in original:
        t, degree, pred = int(r["t"]), int(r["degree"]), float(r["raw_prediction"])
        if not math.isclose(float(r["true_value"]), TRUE_TAIL[t], rel_tol=1e-12, abs_tol=1e-14):
            raise ValueError("Archived benchmark ground truth mismatch.")
        relative = abs(pred - TRUE_TAIL[t]) / TRUE_TAIL[t]
        rows.append(dict(method="NN4MGF_archived", seed="official", degree=degree, t=t,
                         true_value=TRUE_TAIL[t], raw_prediction=pred, relative_error=relative,
                         invalid_prediction=int(not math.isfinite(pred) or pred < 0 or pred > 1),
                         nonfinite_prediction=int(not math.isfinite(pred)),
                         evaluation_origin="archived predictions; NOT re-evaluated"))
    return rows, provenance


def write_csv(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def save_plot(rows, directory):
    colors = {"Cauchy": "tab:blue", "tanh": "tab:orange"}
    styles = {12345: ("-", "o"), 22345: ("--", "s"), 32345: (":", "^")}
    for degree in DEGREES:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        for method in colors:
            for seed in SEEDS:
                selected = sorted([r for r in rows if (r["method"],r["seed"],r["degree"]) == (method,seed,degree)], key=lambda r:r["t"])
                x = [r["t"] for r in selected]
                linestyle, marker = styles[seed]
                axes[0].plot(x, [r["raw_prediction"] for r in selected], marker=marker, markersize=4, linestyle=linestyle, color=colors[method], alpha=0.8, label=f"{method} {seed}")
                axes[1].plot(x, [r["relative_error_percent"] for r in selected], marker=marker, markersize=4, linestyle=linestyle, color=colors[method], alpha=0.8, label=f"{method} {seed}")
        axes[0].plot(T_VALUES, [TRUE_TAIL[t] for t in T_VALUES], "k--", label="Exact tail")
        # symlog preserves finite negative predictions; no clipping or log masking.
        axes[0].set_yscale("symlog", linthresh=1e-4)
        axes[1].set_yscale("symlog", linthresh=1e-3)
        axes[0].set_ylabel("Raw tail prediction (symlog)")
        axes[1].set_ylabel("Relative error (%) (symlog)")
        for ax in axes:
            ax.set_xlabel("t")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=7)
        fig.suptitle(f"20D fixed 25k activation ablation — degree {degree}")
        fig.tight_layout()
        fig.savefig(directory / f"comparison_degree{degree}.png", dpi=150)
        plt.close(fig)


def run_evaluation():
    started = time.perf_counter()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
    output = PACKAGE_DIR / "outputs" / ("evaluation_" + stamp)
    output.mkdir(parents=True, exist_ok=False)  # never overwrite previous results
    rows, audits = [], []
    for entry in CHECKPOINTS:
        print(f"Evaluating {entry['method']} seed {entry['seed']} at step 25000 ...", flush=True)
        model, audit = load_fixed_model(entry)
        rows.extend(evaluate_transform(entry["method"], entry["seed"], learned_tail_transform(model)))
        audit["sha256_after"] = sha256_file(checked_path(entry["path"]))
        if audit["sha256_after"] != audit["sha256_before"]:
            raise RuntimeError("Checkpoint changed during evaluation.")
        audits.append(audit)
        write_csv(output / "pointwise.csv", rows)  # completed endpoints survive interruptions
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
    summary = summarize(rows)
    combined = aggregate(summary)
    floor = evaluate_transform("exact_transform", "analytic", exact_tail_transform)
    archived, provenance = archived_baseline()
    write_csv(output / "summary.csv", summary)
    write_csv(output / "multiseed_summary.csv", combined)
    write_csv(output / "exact_transform_diagnostic.csv", floor)
    write_csv(output / "exact_transform_summary.csv", summarize(floor))
    write_csv(output / "nn4mgf_archived_summary.csv", summarize(archived))
    save_plot(rows, output)
    report = {
        "status": "completed", "training_performed": False, "checkpoint_step": CHECKPOINT_STEP,
        "seeds": list(SEEDS), "degrees": list(DEGREES), "t_values": list(T_VALUES),
        "primary_degree": 8, "degree5_role": "diagnostic and historical comparison",
        "checkpoint_audits": audits, "checkpoint_manifest_sha256": sha256_file(PACKAGE_DIR / "checkpoint_manifest.json"),
        "nn4mgf": provenance, "runtime_seconds": time.perf_counter() - started,
        "python": platform.python_version(), "platform": platform.platform(),
        "torch": str(torch.__version__), "numpy": np.__version__, "mpmath": mp.__version__,
        "device": str(device), "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "complex_dtype": str(c_dtype), "outer_mpmath_dps": 100,
        "talbot_precision": "stock mpmath degree-controlled internal precision; model uses complex128",
        "sampling_budget_per_method_seed": 64_000_000,
        "comparison_scope": "same width/training budget; NOT equal parameter count or identical realized random streams",
        "prediction_clipping": False, "nonfinite_metric_policy": "mark undefined; never drop failed points/seeds",
        "aggregation": "arithmetic mean and sample standard deviation over all three seeds, ddof=1",
        "plot_note": "symlog includes finite negative predictions; nonfinite outcomes remain recorded in pointwise.csv",
    }
    (output / "evaluation_report.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    print("\nDegree 8, interval 1:8 (percent):")
    for row in combined:
        if row["degree"] == 8 and row["interval"] == "1:8":
            print(row)
    print(f"\nCompleted without training. Results: {output}")
    return output


RESULT_DIR = run_evaluation() if RUN_MODE == "evaluate" else None
if RUN_MODE == "check":
    print("Six-checkpoint validation complete. Set RUN_MODE='evaluate' and Run All to compute results.")
